<a href="https://colab.research.google.com/github/13rustam/Zhanakorgan_Agriculture/blob/main/Water_Crop_Use_Kaz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pystac-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.4 MB/s eta 0:00:00


In [6]:
import geopandas as gpd
from pystac_client import Client

In [3]:
catalog = Client.open(
    'https://earth-search.aws.element84.com/v1'
)

In [4]:
for collection in catalog.get_collections():
  print(collection.id)

sentinel-2-pre-c1-l2a
cop-dem-glo-30
naip
cop-dem-glo-90
landsat-c2-l2
sentinel-2-l2a
sentinel-2-l1c
sentinel-2-c1-l2a
sentinel-1-grd


In [7]:
fields_gdf = gpd.read_file('/content/Zhanakorgan_Fields.geojson')
area_gdf = gpd.read_file('/content/Zhanakorgan_Shape_Area.geojson')

In [8]:
print(f'Loaded {len(fields_gdf)} field polygons.')
print(f'Loaded {len(area_gdf)} boundary region(s).')

Loaded 17011 field polygons.
Loaded 1 boundary region(s).


In [9]:
aoi_geometry = area_gdf.geometry.unary_union
bounds = area_gdf.total_bounds.tolist()

/tmp/ipykernel_2144/2531107426.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi_geometry = area_gdf.geometry.unary_union


In [10]:
search = catalog.search(
    collections = ['sentinel-2-c1-l2a'],
    bbox = bounds,
    datetime = '2025'
)

In [11]:
items = search.item_collection()

/usr/local/lib/python3.13/dist-packages/pystac/extensions/storage.py:724: UserWarning: Could not parse bucket/account from href. The following assets were not migrated: ['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail']
  warnings.warn(


In [12]:
len(items)

2023

In [13]:
items[0].properties['proj:code']

'EPSG:32642'

In [14]:
items[0].assets.keys()

dict_keys(['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail'])

In [15]:
!pip install -q stackstac
from stackstac import stack

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 2.7 MB/s eta 0:00:00


In [17]:
layer_stack = stack(
    items,
    assets = ['nir','red','scl'],
    epsg = 32642,
    resolution = 10,
    bounds_latlon = bounds
)

In [18]:
ds = (
    layer_stack
    .to_dataset('band')
    .drop_dims('band')
    .reset_coords(drop = True)
    .drop_attrs()
)

In [19]:
mask = ds.scl.isin([0,1,3,8,9,10,11]).astype(int)
ds_mask = ds[['nir','red']].where(mask == 0)

In [20]:
nir = ds_mask['nir']
red = ds_mask['red']
ndvi = (nir - red) / (nir + red)

In [21]:
ndvi_monthly = ndvi.resample(time = 'ME').max('time')

In [22]:
ndvi_mean = ndvi_monthly.mean(dim = 'time')

In [23]:
from dask.diagnostics import ProgressBar

In [24]:
import warnings
warnings.filterwarnings('ignore', category = RuntimeWarning)
import logging
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
with ProgressBar():
  ndvi_mean = ndvi_mean.compute()

In [ ]:
ndvi_mean.plot(
    robust = True,
    cmap = 'RdYlGn'
)